# Serial DEG analysis

Unlike in the previous notebook, here we will try including all pairs of samples into the model
to better model dispersions.

In [2]:
import sys
sys.path.insert(0, '../lib')

In [3]:
import os
import pandas as pd

import common_data

In [4]:
pd.options.display.max_columns = 200
pd.options.display.max_rows = 200
pd.options.display.max_colwidth = 1000
%config InlineBackend.figure_format = "retina"

In [5]:
def sanitize_name(name):
    return name.replace(' ', '_').replace('*', '').replace(';', '_and').replace('/', '_')

# Save serial DEG tasks

In [ ]:
BASE = common_data.DATA / '05_pseudobulk/40a_serial'

In [7]:
os.makedirs(BASE, exist_ok=True)

Repeating code from 09_metadata/07_serial

In [8]:
df = pd.read_csv(common_data.SC_LABELS, index_col=0)
labels = common_data.get_sc_categorical_covariates()

In [9]:
df = df.loc[df.cohort.eq('SCRIPT')]

In [10]:
df = df.merge(labels, on='bal_barcode', suffixes=('', '_label'), how='left')

In [12]:
bals_per_pt = df.groupby('Patient_id').size()

In [13]:
df = df.loc[df.Patient_id.isin(bals_per_pt.index[bals_per_pt.gt(1)])]

In [14]:
df.Patient_id.nunique()

55

In [15]:
bals_per_pt.loc[bals_per_pt.gt(1)].value_counts().sort_index()

2    34
3    14
4     2
5     2
6     2
8     1
dtype: int64

Explore all patients

In [ ]:
df.sort_values(['Patient_id', 'day_of_hospitalization']).groupby('Patient_id').apply(
    lambda x: '|'.join((x.day_of_hospitalization.astype(int).astype(str) + ':' + x.episode_type))
)

## VAP and no VAP pairs of BALs

For each patient, we will find a pair of samples that interests us:
1. Transitions to VAP (which can be HAP with superinfection) from CAP/HAP/NPC within 14 days
2. If not found, any other pair within 14 days

In [38]:
DAYS_BETWEEN_THRESHOLD = 14

rejected_pairs = 0

pairs = []
for patient in df.Patient_id.unique():
    patient_df = df.loc[df.Patient_id.eq(patient)]
    found = False
    for i in range(0, len(patient_df) - 1):
        first = patient_df.iloc[i]
        second = patient_df.iloc[i + 1]
        day_diff = second.day_of_hospitalization - first.day_of_hospitalization
        if day_diff <= DAYS_BETWEEN_THRESHOLD:
            # Determine the label
            if (first.episode_type in ('CAP', 'HAP', 'NPC')
                    and second.episode_type in ('CAP', 'HAP', 'NPC', 'none')
                    and second.future_vap_onset != 'already_started'):
                pairs.append((first.bal_barcode, second.bal_barcode, 'NO VAP'))
                found = True
                break
            if (first.episode_type in ('CAP', 'HAP', 'NPC')
                    and first.future_vap_onset != 'already_started'
                    and (second.episode_type == 'VAP'
                        or (second.episode_type == 'HAP'
                            and second.future_vap_onset == 'already_started'))):
                pairs.append((first.bal_barcode, second.bal_barcode, 'VAP'))
                found = True
                break
            pairs.append((first.bal_barcode, second.bal_barcode, 'discard'))
            found = True
            break
    if not found:
        print(f"No suitable pair found for patient {patient} with {len(patient_df)} samples")
        rejected_pairs += 1

print(f"Rejected {rejected_pairs} pairs")


No suitable pair found for patient 9238.0 with 2 samples
No suitable pair found for patient 1382.0 with 2 samples
No suitable pair found for patient 3419.0 with 2 samples
No suitable pair found for patient 4623.0 with 2 samples
No suitable pair found for patient 4419.0 with 2 samples
No suitable pair found for patient 8601.0 with 2 samples
Rejected 6 pairs


In [39]:
pairs = pd.DataFrame(pairs, columns=['first_bal', 'second_bal', 'label'])
pairs = pairs.merge(
    df[['bal_barcode', 'Episode_id', 'episode_type', 'Pathogen_groups', 'day_of_hospitalization', 'future_vap_onset']],
    left_on='first_bal',
    right_on='bal_barcode',
    how='left'
)
pairs = pairs.merge(
    df[['bal_barcode', 'Episode_id', 'episode_type', 'Pathogen_groups', 'day_of_hospitalization', 'future_vap_onset', 'vap_onset']],
    left_on='second_bal',
    right_on='bal_barcode',
    how='left',
    suffixes=('', '_2')
)
pairs.insert(2, 'days_between', pairs.day_of_hospitalization_2 - pairs.day_of_hospitalization)
pairs.drop(columns=['bal_barcode', 'bal_barcode_2', 'day_of_hospitalization', 'day_of_hospitalization_2'], inplace=True)

In [40]:
pairs.shape

(49, 13)

In [41]:
pairs[['episode_type', 'episode_type_2', 'label']].value_counts(dropna=False).reset_index()

,episode_type,episode_type_2,label,0
0,VAP,VAP,discard,15
1,HAP,HAP,NO VAP,7
2,CAP,CAP,NO VAP,4
3,HAP,VAP,VAP,4
4,HAP,none,NO VAP,3
5,NPC,VAP,VAP,3
6,VVAP,VAP,discard,3
7,HAP,HAP,VAP,2
8,CAP,VAP,VAP,1
9,NPC,NPC,NO VAP,1


In [42]:
pairs[['episode_type', 'future_vap_onset', 'episode_type_2', 'future_vap_onset_2', 'label']].value_counts(dropna=False).reset_index()

,episode_type,future_vap_onset,episode_type_2,future_vap_onset_2,label,0
0,VAP,already_started,VAP,already_started,discard,14
1,VVAP,viral_only_vap,VAP,viral_only_vap,discard,3
2,HAP,no_vap,HAP,no_vap,NO VAP,3
3,HAP,no_vap,none,no_vap,NO VAP,3
4,CAP,no_vap,CAP,no_vap,NO VAP,2
5,HAP,starts_later,HAP,starts_later,NO VAP,2
6,NPC,day_-5,VAP,already_started,VAP,1
7,none,already_started,VAP,already_started,discard,1
8,VVAP,viral_only_vap,none,viral_only_vap,discard,1
9,VVAP,viral_only_vap,VVAP,viral_only_vap,discard,1


In [43]:
pairs_combined = pairs.copy()

In [44]:
labels = common_data.get_sc_categorical_covariates()

In [45]:
idx = pairs.label.eq('VAP')
labels.loc[pairs.first_bal[idx], 'group'] = 'vap'
labels.loc[pairs.first_bal[idx], 'timepoint'] = 'first'
labels.loc[pairs.first_bal[idx], 'pair_id'] = pairs.first_bal[idx].values
labels.loc[pairs.second_bal[idx], 'group'] = 'vap'
labels.loc[pairs.second_bal[idx], 'timepoint'] = 'second'
labels.loc[pairs.second_bal[idx], 'pair_id'] = pairs.first_bal[idx].values

idx = pairs.label.eq('NO VAP')
labels.loc[pairs.first_bal[idx], 'group'] = 'no_vap'
labels.loc[pairs.first_bal[idx], 'timepoint'] = 'first'
labels.loc[pairs.first_bal[idx], 'pair_id'] = pairs.first_bal[idx].values
labels.loc[pairs.second_bal[idx], 'group'] = 'no_vap'
labels.loc[pairs.second_bal[idx], 'timepoint'] = 'second'
labels.loc[pairs.second_bal[idx], 'pair_id'] = pairs.first_bal[idx].values

idx = pairs.label.eq('discard')
labels.loc[pairs.first_bal[idx], 'group'] = 'discard'
labels.loc[pairs.first_bal[idx], 'timepoint'] = 'first'
labels.loc[pairs.first_bal[idx], 'pair_id'] = pairs.first_bal[idx].values
labels.loc[pairs.second_bal[idx], 'group'] = 'discard'
labels.loc[pairs.second_bal[idx], 'timepoint'] = 'second'
labels.loc[pairs.second_bal[idx], 'pair_id'] = pairs.first_bal[idx].values

In [46]:
labels.group.value_counts()

discard    46
no_vap     32
vap        20
Name: group, dtype: int64

In [47]:
pairs.shape[0] * 2

98

In [ ]:
os.makedirs(BASE, exist_ok=True)
labels.loc[
    labels.group.isin(['vap', 'no_vap', 'discard']),
    ['group', 'timepoint', 'pair_id']
].reset_index().to_csv(
    BASE / '_group_labels.csv',
)

In [55]:
pairs_combined.loc[pairs_combined.label.eq('NO VAP')].Pathogen_groups.astype(str).value_counts()

Early SARS-CoV-2           9
discard                    4
Late SARS-CoV-2            1
NPC                        1
Early SARS-CoV-2; Gram+    1
Name: Pathogen_groups, dtype: int64

In [56]:
pairs_combined.loc[pairs_combined.label.eq('VAP')].Pathogen_groups.astype(str).value_counts()

Early SARS-CoV-2          3
NPC                       3
Late SARS-CoV-2; Gram+    1
Late SARS-CoV-2           1
discard                   1
Gram-*; Gram+             1
Name: Pathogen_groups, dtype: int64

In [63]:
df.loc[df.bal_barcode.isin(pairs_combined.first_bal[pairs_combined.label.eq('NO VAP')])].day_of_hospitalization.describe()

count    16.000000
mean      5.000000
std       4.898979
min       1.000000
25%       1.750000
50%       3.000000
75%       7.250000
max      17.000000
Name: day_of_hospitalization, dtype: float64

In [64]:
df.loc[df.bal_barcode.isin(pairs_combined.first_bal[pairs_combined.label.eq('VAP')])].day_of_hospitalization.describe()

count    10.000000
mean      9.500000
std      10.834615
min       2.000000
25%       3.000000
50%       6.000000
75%      10.250000
max      38.000000
Name: day_of_hospitalization, dtype: float64